# Bonus Lab: FireDucks

**Module:** Week 3 Day 1 (optional)  
**Kernel:** `.venv-probes` (build it first with `Bonus_Lab_Engine_Probes.md`)


> This is an optional bonus walkthrough. You are not expected to master this tool today.
> The goal is awareness: understand what problem this tool tries to solve, when it may help, and what trade-offs it introduces.


FireDucks also keeps the pandas API: `import fireducks.pandas as fpd`. Underneath it compiles your operations and, like Polars lazy and Dask, it can defer work.

**Why a separate kernel?** FireDucks is environment-sensitive. At the time of writing, PyPI provides wheels for Linux x86_64 and macOS ARM64, not Windows. Keep it in `.venv-probes` so it does not disturb the main course environment.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REAL_PATH = Path("data/yellow_tripdata_2024-04.parquet")
SYNTH_PATH = Path("data/synthetic_tripdata.parquet")

if REAL_PATH.exists():
    SOURCE_PATH = REAL_PATH
    print("using real TLC data:", SOURCE_PATH)
else:
    if not SYNTH_PATH.exists():
        rng = np.random.default_rng(42)
        n = 1_000_000
        pickup = pd.to_datetime("2024-04-01") + pd.to_timedelta(
            rng.integers(0, 30 * 24 * 3600, n), unit="s"
        )
        duration_s = rng.integers(60, 5400, n)
        synth = pd.DataFrame(
            {
                "tpep_pickup_datetime": pickup,
                "tpep_dropoff_datetime": pickup + pd.to_timedelta(duration_s, unit="s"),
                "payment_type": rng.integers(0, 5, n),
                "fare_amount": (duration_s / 60 * rng.uniform(2.0, 4.0, n)).round(2),
                "trip_distance": (duration_s / 60 * rng.uniform(0.2, 0.6, n)).round(2),
            }
        )
        SYNTH_PATH.parent.mkdir(exist_ok=True)
        synth.to_parquet(SYNTH_PATH, index=False)
    SOURCE_PATH = SYNTH_PATH
    print("real file not found, using synthetic data:", SOURCE_PATH)

In [ ]:
import importlib.util

fireducks_available = importlib.util.find_spec("fireducks") is not None

if not fireducks_available:
    print("FireDucks is not installed in this kernel.")
    print("Install attempt: uv pip install --python .venv-probes fireducks")
    print("If this fails, skip this optional notebook and record the OS/Python/platform reason.")
else:
    import fireducks.pandas as fpd
    print("FireDucks ready")


In [ ]:
%%time
if fireducks_available:
    fdf = fpd.read_parquet(SOURCE_PATH)
    fdf["trip_duration_min"] = (
        (fdf["tpep_dropoff_datetime"] - fdf["tpep_pickup_datetime"]).dt.total_seconds() / 60
    )
    fireducks_answer = fdf[fdf["trip_duration_min"] > 25].groupby("payment_type")["fare_amount"].mean()
    print(fireducks_answer)  # printing forces evaluation, so the timing is honest
else:
    print("FireDucks timing skipped.")

That deferral is the benchmarking catch: a cell can look instant because nothing was computed yet. To time FireDucks honestly, the timed block must end with something that forces the result to exist, like printing the answer.

## Your Turn
Change the business question: **for trips longer than 10 minutes, what is the average fare by payment type?**

In [ ]:
# Your code here

## Bonus Tool Takeaways

| Topic | What students should remember | Do they need it today? |
|---|---|---|
| Dask | Pandas-like API, lazy execution, partitions, `.compute()` | No, awareness only |
| Modin | Tries to keep Pandas code while changing the execution engine | No, awareness only |
| FireDucks | Pandas-like acceleration, but environment-sensitive | No, awareness only |
| Parquet | Columnar storage, smaller files, column pruning, compression trade-offs | Yes, important for data engineering |
